# 0. Install and Import Dependencies

In [ ]:
%pip list

In [ ]:
%pip install opencv-python matplotlib imageio gdown tensorflow

In [5]:
import os
import cv2
import tensorflow as tf
import keras
import numpy as np
from typing import List
from matplotlib import pyplot as plt
import imageio

## Project Overview

This notebook implements a lip reading model that transcribes spoken words 
from silent video of a speaker's lips — no audio required.

**Dataset:** GRID Corpus (Speaker S1) — ~500 videos of a single speaker 
reciting sentences in a fixed pattern:
`[command] [color] [preposition] [letter] [digit] [adverb]`

**Why CTC Loss?** Standard cross-entropy requires knowing which input frame 
corresponds to which output character at every timestep — an alignment problem 
that's impractical to label manually. CTC solves this by summing probabilities 
over all valid alignments during training, so only the final transcript is needed 
as a label.

**Why 75 frames?** GRID videos are 25fps × 3 seconds = 75 frames. The 3D CNN 
requires a fixed input shape, so all videos are padded or truncated to exactly 
75 frames.

**Why crop [190:236, 80:220]?** These coordinates isolate the lip region in 
GRID corpus videos, which have consistent speaker positioning. The crop produces 
a 46×140 pixel grayscale region.

In [ ]:
print(tf.__version__)
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
try:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
except:
    pass

# 1. Build Data Loading Functions

In [ ]:
# Training setup:
# If running on Google Colab with GPU:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   CHECKPOINT_PATH = '/content/drive/MyDrive/LipNet/checkpoint.weights.h5'
# If running locally:
CHECKPOINT_PATH = 'models/checkpoint.weights.h5'

In [ ]:
def load_video(path: str) -> List[float]:
    """
    Extracts and preprocesses frames from a video file.
    
    Steps:
    - Reads all frames using OpenCV
    - Converts to grayscale using cv2 (avoids TensorFlow reshape issues with variable frame sizes)
    - Crops lip region: rows 190-236, cols 80-220 → 46×140 pixels
    - Resizes to exactly 46×140 to handle any resolution inconsistencies
    - Pads with black frames or truncates to exactly 75 frames
    - Normalizes: (frames - mean) / std across all frames
    
    Args:
        path: Path to .mpg video file
        
    Returns:
        Float32 numpy array of shape (75, 46, 140, 1)
    """
    cap = cv2.VideoCapture(path)
    frames = []
    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))): 
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = frame[190:236, 80:220]
        frame = cv2.resize(frame, (140, 46))
        frame = np.expand_dims(frame, axis=-1)
        frames.append(frame)
    cap.release()

    while len(frames) < 75:
        frames.append(np.zeros((46, 140, 1), dtype=np.float32))
    frames = frames[:75]

    frames = np.array(frames, dtype=np.float32)
    mean = frames.mean()
    std = frames.std()
    std = max(std, 1e-6)
    return (frames - mean) / std

In [ ]:
url = 'https://drive.google.com/uc?id=1YlvpDLix3S-U8fd-gqRwPcWXAXm8JwjL'
output = 'data.zip'
gdown.download(url, output, quiet=False)
gdown.extractall('data.zip')

In [11]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

In [ ]:
char_to_num = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="")
num_to_char = tf.keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True
)

print(
    f"The vocabulary is: {char_to_num.get_vocabulary()} "
    f"(size ={char_to_num.vocabulary_size()})"
)

In [ ]:
char_to_num.get_vocabulary()

In [ ]:
char_to_num(['n','i','c','k'])

In [ ]:
num_to_char([14,  9,  3, 11])

In [16]:
def load_alignments(path:str) -> List[str]:
    with open(path, 'r') as f:
        lines = f.readlines()
    tokens = []
    for line in lines:
        line = line.split()
        if len(line) >= 3 and line[2] != 'sil':
            tokens = [*tokens,' ',line[2]]

    if not tokens:
        tokens = [' ']  # fallback for empty alignments

    result = char_to_num(tf.reshape(tf.strings.unicode_split(tokens, input_encoding='UTF-8'), (-1)))
    return result[1:] if len(result) > 1 else result

In [17]:
def load_data(path: str):
    path = bytes.decode(path.numpy())
    file_name = os.path.splitext(os.path.basename(path))[0]
    video_path = os.path.join('data','s1',f'{file_name}.mpg')
    alignment_path = os.path.join('data','alignments','s1',f'{file_name}.align')
    frames = load_video(video_path)
    alignments = load_alignments(alignment_path)

    return frames, alignments

In [18]:
test_path = './data/s1/bbal6n.mpg'

In [ ]:
tf.convert_to_tensor(test_path).numpy().decode('utf-8').split('/')[-1].split('.')[0]

In [20]:
frames, alignments = load_data(tf.convert_to_tensor(test_path))

In [ ]:
plt.imshow(frames[40])

In [ ]:
alignments

In [ ]:
tf.strings.reduce_join([bytes.decode(x) for x in num_to_char(alignments.numpy()).numpy()])

In [24]:
def mappable_function(path:str) ->List[str]:
    result = tf.py_function(load_data, [path], (tf.float32, tf.int64))
    return result

# 2. Create Data Pipeline

In [25]:
from matplotlib import pyplot as plt

In [26]:
data = tf.data.Dataset.list_files('./data/s1/*.mpg')
data = data.shuffle(500, reshuffle_each_iteration=False)
data = data.map(mappable_function)
data = data.padded_batch(2, padded_shapes=([75,46,140,1],[40]))
data = data.prefetch(tf.data.AUTOTUNE)
# Added for split
train = data.take(450)
test = data.skip(450)

In [ ]:
len(test)

In [28]:
frames, alignments = data.as_numpy_iterator().next()

In [29]:
len(frames)

2

In [30]:
sample = data.as_numpy_iterator()

In [ ]:
val = sample.next(); val[0]

In [32]:
#imageio.mimsave('./animation.gif', val[0][0], fps=10)

In [ ]:
# 0:videos, 0: 1st video out of the batch,  0: return the first frame in the video
plt.imshow(val[0][0][60])

In [ ]:
tf.strings.reduce_join([num_to_char(word) for word in val[1][0]])

# 3. Design the Deep Neural Network

## Model Architecture

The architecture follows the LipNet paper: 3D CNNs extract spatiotemporal 
features from the video, TimeDistributed(Flatten) converts each frame's 
feature map into a vector, and Bidirectional LSTMs model the temporal 
sequence for CTC decoding.

Filter progression: 128 → 256 → 75 (from LipNet paper)
Each MaxPool3D(1,2,2) halves spatial dimensions but preserves the 75 time steps.

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, LSTM, Dense, Dropout, Bidirectional, MaxPool3D, Activation, Reshape, SpatialDropout3D, BatchNormalization, TimeDistributed, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler

In [ ]:
data.as_numpy_iterator().next()[0][0].shape

In [ ]:
model = Sequential()
model.add(Conv3D(128, 3, input_shape=(75,46,140,1), padding='same'))
model.add(Activation('relu'))
model.add(MaxPool3D((1,2,2)))

model.add(Conv3D(256, 3, padding='same'))
model.add(Activation('relu'))
model.add(MaxPool3D((1,2,2)))

model.add(Conv3D(75, 3, padding='same'))
model.add(Activation('relu'))
model.add(MaxPool3D((1,2,2)))

model.add(TimeDistributed(Flatten()))

model.add(Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True)))
model.add(Dropout(.5))

model.add(Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True)))
model.add(Dropout(.5))

model.add(Dense(char_to_num.vocabulary_size()+1, kernel_initializer='he_normal', activation='softmax'))

In [ ]:
model.summary()

In [ ]:
yhat = model.predict(val[0])

In [ ]:
tf.strings.reduce_join([num_to_char(x) for x in tf.argmax(yhat[0],axis=1)])

In [ ]:
tf.strings.reduce_join([num_to_char(tf.argmax(x)) for x in yhat[0]])

In [ ]:
model.input_shape

In [ ]:
model.output_shape

# 4. Setup Training Options and Train

In [45]:
def scheduler(epoch, lr):
    if epoch < 30:
        return lr
    else:
        return float(lr * tf.math.exp(-0.1))

In [46]:
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

In [47]:
class ProduceExample(tf.keras.callbacks.Callback):
    def __init__(self, dataset) -> None:
        self.dataset = dataset.as_numpy_iterator()

    def on_epoch_end(self, epoch, logs=None) -> None:
        data = self.dataset.next()
        yhat = self.model.predict(data[0])
        decoded = tf.keras.backend.ctc_decode(yhat, [75,75], greedy=False)[0][0].numpy()
        for x in range(len(yhat)):
            print('Original:', tf.strings.reduce_join(num_to_char(data[1][x])).numpy().decode('utf-8'))
            print('Prediction:', tf.strings.reduce_join(num_to_char(decoded[x])).numpy().decode('utf-8'))
            print('~'*100)

In [48]:
class MetricsLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        loss = logs.get('loss')
        val_loss = logs.get('val_loss')
        print(f"Epoch {epoch+1:03d} | Loss: {loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
checkpoint_callback = ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor='val_loss',
    save_weights_only=True
)

In [50]:
schedule_callback = LearningRateScheduler(scheduler)

In [51]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss=CTCLoss)

In [52]:
example_callback = ProduceExample(test)

In [ ]:
import os
os.listdir('/content/drive/MyDrive/LipNet/')

In [57]:
model.load_weights('/content/drive/MyDrive/LipNet/checkpoint.weights.h5')

In [58]:
example_callback = ProduceExample(test)
metrics_callback = MetricsLogger()

In [ ]:
model.fit(
    train,
    validation_data=test,
    epochs=100,
    initial_epoch=89,
    callbacks=[checkpoint_callback, schedule_callback, example_callback, metrics_callback]
)

In [ ]:
# ── Word Error Rate Evaluation ──────────────────────────────────────────────
# WER measures the edit distance between predicted and actual word sequences,
# normalized by the number of words in the actual transcript.
# Lower is better. Human-level WER on GRID corpus is ~0%.

def calculate_wer(actual: str, predicted: str) -> float:
    """
    Calculates Word Error Rate using dynamic programming (edit distance).
    WER = (substitutions + insertions + deletions) / total words in reference
    """
    a = actual.split()
    p = predicted.split()
    d = np.zeros((len(a)+1, len(p)+1))
    for i in range(len(a)+1): d[i][0] = i
    for j in range(len(p)+1): d[0][j] = j
    for i in range(1, len(a)+1):
        for j in range(1, len(p)+1):
            cost = 0 if a[i-1] == p[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(a)][len(p)] / len(a)

total_wer = []
for frames, alignments in list(test.as_numpy_iterator())[:20]:
    for i in range(len(frames)):
        yhat = model.predict(tf.expand_dims(frames[i], axis=0), verbose=0)
        decoded = tf.keras.backend.ctc_decode(yhat, [75], greedy=True)[0][0].numpy()
        actual = tf.strings.reduce_join([num_to_char(w) for w in alignments[i]]).numpy().decode('utf-8').strip()
        pred = tf.strings.reduce_join([num_to_char(w) for w in decoded[0]]).numpy().decode('utf-8').strip()
        total_wer.append(calculate_wer(actual, pred))

print(f"Average WER on test set: {np.mean(total_wer):.2%}")
print(f"Samples evaluated: {len(total_wer)}")

Average WER: 11.61%


# 5. Make a Prediction

In [ ]:
url = 'https://drive.google.com/uc?id=1vWscXs4Vt0a_1IH1-ct2TCgXAZT-N3_Y'
output = 'checkpoints.zip'
gdown.download(url, output, quiet=False)
gdown.extractall('checkpoints.zip', 'models')

In [ ]:
import gdown

# Download pre-trained weights from Google Drive
WEIGHTS_URL = "https://drive.google.com/file/d/1kHs4JdQWWL3iSmm-CFwMERycRwDB_vap/view?usp=sharing"
CHECKPOINT_PATH = "models/checkpoint.weights.h5"

import os
os.makedirs("models", exist_ok=True)
if not os.path.exists(CHECKPOINT_PATH):
    gdown.download(WEIGHTS_URL, CHECKPOINT_PATH, quiet=False)

model.load_weights(CHECKPOINT_PATH)

In [71]:
test_data = test.as_numpy_iterator()

In [72]:
sample = test_data.next()

In [ ]:
yhat = model.predict(sample[0])

In [ ]:
print('~'*100, 'REAL TEXT')
[tf.strings.reduce_join([num_to_char(word) for word in sentence]) for sentence in sample[1]]

In [75]:
decoded = tf.keras.backend.ctc_decode(yhat, input_length=[75,75], greedy=True)[0][0].numpy()

In [ ]:
print('~'*100, 'PREDICTIONS')
[tf.strings.reduce_join([num_to_char(word) for word in sentence]) for sentence in decoded]

# Test on a Video

In [77]:
sample = load_data(tf.convert_to_tensor('./data/s1/bras9a.mpg'))

In [ ]:
print('~'*100, 'REAL TEXT')
[tf.strings.reduce_join([num_to_char(word) for word in sentence]) for sentence in [sample[1]]]

In [ ]:
yhat = model.predict(tf.expand_dims(sample[0], axis=0))

In [80]:
decoded = tf.keras.backend.ctc_decode(yhat, input_length=[75], greedy=True)[0][0].numpy()

In [ ]:
print('~'*100, 'PREDICTIONS')
[tf.strings.reduce_join([num_to_char(word) for word in sentence]) for sentence in decoded]

In [84]:
test_samples = list(test.as_numpy_iterator())

for i in range(min(5, len(test_samples))):
    frames, alignments = test_samples[i]

    yhat = model.predict(tf.expand_dims(frames[0], axis=0))
    decoded = tf.keras.backend.ctc_decode(yhat, input_length=[75], greedy=True)[0][0].numpy()

    real = tf.strings.reduce_join([num_to_char(w) for w in alignments[0]]).numpy().decode('utf-8').strip()
    pred = tf.strings.reduce_join([num_to_char(w) for w in decoded[0]]).numpy().decode('utf-8').strip()

    print(f"Real:      {real}")
    print(f"Predicted: {pred}")
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
Real:      place green by y five soon
Predicted: place green by y five soon

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
Real:      bin red by m six now
Predicted: bin red by v six now

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Real:      lay blue at y zero please
Predicted: lay blue at y zero please

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
Real:      bin green with b five soon
Predicted: bin green with b five soon

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
Real:      set green in o seven soon
Predicted: set green in s seven soon



In [85]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss=CTCLoss)